<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/11-catalyst-optimizer/01-explain-modes.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 11.1 — explain() modes: five doors into the plan

`simple`, `extended`, `cost`, `formatted`, `codegen` — each answers a
different question. Run them all on one query, then practice the
four-step reading routine (count Exchanges → check joins → read the
scan → worry about codegen last).

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("11.1-explain-modes")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
big   = spark.range(2_000_000).withColumn("k", (col("id") % 100).cast("int")) \
                                  .withColumn("v", col("id") * 2)
small = spark.range(100).withColumnRenamed("id", "k")   # tiny — broadcastable
q = big.filter(col("v") > 100).join(small, "k").groupBy("k").sum("v")

## `simple` (the default): what will actually run

Just the physical plan. Your day-to-day mode. Read bottom-up. Count
the `Exchange` nodes; find the join operator.

In [ ]:
q.explain()                 # == q.explain(mode="simple")

## `extended`: all four plans stacked (the 11.0 pipeline)

parsed → analyzed → optimized logical → physical. Use it to see *why*
the physical plan came out the way it did.

In [ ]:
q.explain(mode="extended")   # legacy: q.explain(True) is the same

## `formatted`: numbered tree on top, per-node detail below

The most readable mode for anything non-trivial. The detail blocks
hold `PushedFilters`, `ReadSchema`, join keys — the fine print.

In [ ]:
q.explain(mode="formatted")

## `cost`: the optimized plan annotated with statistics

`sizeInBytes` always; `rowCount` only when stats exist. This is what
the join planner decides from.

In [ ]:
q.explain(mode="cost")

## `codegen`: the generated Java (skim it)

Definitive proof of what got compiled into a loop (lesson 11.3). Long
— just confirm it's Java source, don't read every line.

In [ ]:
# scan+filter+project compile into one loop — this dumps that Java
big.filter(col("v") > 100).select("k").explain(mode="codegen")

## The plan as data, not text

`queryExecution` gives you the trees programmatically — useful for
tests ("assert this plan has no Exchange").

In [ ]:
plan_str = big.filter(col("v") > 100).select("k").queryExecution.executedPlan.toString()
print("Exchanges in this plan:", plan_str.count("Exchange"))   # narrow query -> 0

plan_str2 = q.queryExecution.executedPlan.toString()
print("Exchanges in the join+agg query:", plan_str2.count("Exchange"))

In [ ]:
spark.stop()

## Exercises

1. Run the four-step routine on `q`'s `simple` plan out loud: (a) how
   many Exchanges? (b) is the join broadcast or sort-merge? (c) what's
   in the scan's PushedFilters/ReadSchema? (d) which nodes have `*`?
2. Which mode answers each: "did this become a broadcast join?",
   "why did my `WHERE 1=1` disappear?", "what row count does Spark
   estimate?", "did my subtree get compiled?"
3. `explain(True)` and `explain(mode="extended")` — confirm they print
   the same thing. What about `explain()` vs `explain(mode="simple")`?
4. Take the join query and force `SortMergeJoin` by disabling
   broadcast (`spark.conf.set("spark.sql.autoBroadcastJoinThreshold",
   -1)`). Re-run `simple`. How did the Exchange count change?
5. Write a one-line assertion helper `has_no_shuffle(df)` using
   `queryExecution.executedPlan.toString()`. Test it on a narrow query
   and a groupBy.

In [ ]:
# your exercise code here